In [ ]:
# IMPORTS
import pandas as pd
import xgboost as xgb
from emag_reports.logger import logger
from emag_reports.discovery.attribute_reduction import feature_importance_xgboost
from emag_reports.discovery.indicators import calculate_classification_indicators as calculate_indicators
from emag_reports.discovery.itemset_mining import find_frequent_itemsets_classification as find_frequent_itemsets
from emag_reports.discovery.rules import CN2Model
from emag_reports.discovery.trees import build_and_show_classification_decision_tree
from emag_reports.preprocess import balance_decision_classes
from emag_reports.preprocess import remove_low_correlation_attributes
from emag_reports.preprocess import encode_dataset
from emag_reports.preprocess import decode_dataset
from emag_reports.preprocess import impute_missing_values
from emag_reports.preprocess import remove_low_quality_attributes
from IPython.display import display
from IPython.display import HTML
from IPython.display import Markdown as md
from itables import show
from mlxtend.frequent_patterns import association_rules

In [ ]:
## DEFAULT PARAMETERS
# DATASET PARAMETERS
TITLE = "Dataset exploration - knowledge discovery (whitebox models)"
NAME = "diabetes"
CLASS_ATTRIBUTE = "class"
# DATA PREPARATION
# Unimportant attributes elimination
UNIMPORTANT_ATTRIBUTE_ELIMINATION = True
USE_LOW_CORRELATION_ATTRIBUTE_REMOVAL = True
I = 0.9
S = 0.9
M = 0.9
T = 0.9
# Missing values imputation
MISSING_VALUES_IMPUTATION = True
# Strategy
STRATEGY_NUMERICAL = 'mean'
STRATEGY_NOMINAL = 'most_frequent'
# Decision class balancing
DECISION_CLASS_BALANCING = True
# MODELS
# Trees
TREES = True
MAX_DEPTH = 5
# Rules
RULES = True
MIN_COVERED_EXAMPLES = 5
MAX_RULE_LENGTH = 5
BEAM_WIDTH = 10
# Itemset mining and associations
ITEMSETS_AND_ASSOCIATION = True
EXCLUDE_TARGET_COLUMN = False
MAX_K_VALUE = 3
MAX_INTERVALS_VALUE = 10
MIN_CONFIDENCE = 0.5
MIN_SUPP = 0.01

DATASET = None

MAX_ROWS = 500

In [ ]:
html_content = f"""
<div style='text-align: center; margin-bottom: 30px; font-family: Arial, sans-serif;'>
    <h1 style='font-size: 2em; color: #2c3e50;'><strong>Knowledge discovery</strong></h1>
    <h2 style='font-size: 1.3em; color: #34495e;'>Extract interpretable patterns from data</h2>
</div>
<div style='margin: 0 auto; width: 90%; font-family: Arial, sans-serif; color: #2c3e50;'>
    <section style='margin-bottom: 30px;'>
        <h2 style='border-bottom: 2px solid #7f8c8d; padding-bottom: 10px;'>Report information</h2>
        <div style='padding-left: 2em; line-height: 1.5;'>
            <p><strong>Report name:</strong> <span style='font-style: italic; color: #2980b9;'>{TITLE}</span></p>
            <p><strong>Dataset name:</strong> <span style='font-style: italic; color: #2980b9;'>{NAME}</span></p>
        </div>
    </section>
    <section style='margin-bottom: 30px;'>
        <h3 style='border-bottom: 1px solid #bdc3c7; padding-bottom: 5px;'>Data preprocessing parameters</h3>
        <div style='padding-left: 2em; line-height: 1.5;'>
            <p>Remove attributes of low quality: <span style='font-style: italic; color: #2980b9;'>{UNIMPORTANT_ATTRIBUTE_ELIMINATION}</span></p>"""

if UNIMPORTANT_ATTRIBUTE_ELIMINATION:
    html_content += f"""
            <div style='padding-left: 2em;'>
                <p>Remove attributes weakly correlated with the decision attribute: <span style='font-style: italic; color: #2980b9;'>{USE_LOW_CORRELATION_ATTRIBUTE_REMOVAL}</span></p>
                <p>Excessive number of unique values threshold: <span style='font-style: italic; color: #2980b9;'>{I}</span></p>
                <p>Stability threshold: <span style='font-style: italic; color: #2980b9;'>{S}</span></p>
                <p>Missing values threshold: <span style='font-style: italic; color: #2980b9;'>{M}</span></p>
                <p>Text-like - the variable values resemble text threshold: <span style='font-style: italic; color: #2980b9;'>{T}</span></p>
            </div>"""

html_content += f"""
            <p>Replace missing values: <span style='font-style: italic; color: #2980b9;'>{MISSING_VALUES_IMPUTATION}</span></p>"""

if MISSING_VALUES_IMPUTATION:
    html_content += f"""
            <div style='padding-left: 2em;'>
                <p>For numeric attributes by: <span style='font-style: italic; color: #2980b9;'>{STRATEGY_NUMERICAL}</span></p>
                <p>For nominal attributes by: <span style='font-style: italic; color: #2980b9;'>{STRATEGY_NOMINAL}</span></p>
            </div>"""

html_content += f"""
            <p>Balance the sizes of decision classes: <span style='font-style: italic; color: #2980b9;'>{DECISION_CLASS_BALANCING}</span></p>
        </div>
    </section>"""

if TREES:
    html_content += f"""
    <section style='margin-bottom: 30px;'>
        <h3 style='border-bottom: 1px solid #bdc3c7; padding-bottom: 5px;'>Tree parameters</h3>
        <div style='padding-left: 2em; line-height: 1.5;'>
            <p>Maximum depth: <span style='font-style: italic; color: #2980b9;'>{MAX_DEPTH}</span></p>
        </div>
    </section>
    """

if RULES:
    html_content += f"""
    <section style='margin-bottom: 30px;'>
        <h3 style='border-bottom: 1px solid #bdc3c7; padding-bottom: 5px;'>Rule parameters</h3>
        <div style='padding-left: 2em; line-height: 1.5;'>
            <p>CN2:</p>
            <div style='padding-left: 2em;'>
                <p>Minimum number of covered examples: <span style='font-style: italic; color: #2980b9;'>{MIN_COVERED_EXAMPLES}</span></p>
                <p>Maximum number of elementary conditions: <span style='font-style: italic; color: #2980b9;'>{MAX_RULE_LENGTH}</span></p>
                <p>Beam width: <span style='font-style: italic; color: #2980b9;'>{BEAM_WIDTH}</span></p>
            </div>
        </div>
    </section>
    """

if ITEMSETS_AND_ASSOCIATION:
    html_content += f"""
    <section style='margin-bottom: 30px;'>
        <h3 style='border-bottom: 1px solid #bdc3c7; padding-bottom: 5px;'>Frequent itemsets and association rules parameters</h3>
        <div style='padding-left: 2em; line-height: 1.5;'>
            <p>Exclude the decision attribute: <span style='font-style: italic; color: #2980b9;'>{EXCLUDE_TARGET_COLUMN}</span></p>
            <p>Maximum number of intervals for numeric attributes: <span style='font-style: italic; color: #2980b9;'>{MAX_INTERVALS_VALUE}</span></p>
            <p>Maximum length: <span style='font-style: italic; color: #2980b9;'>{MAX_K_VALUE}</span></p>
            <p>Minimum support: <span style='font-style: italic; color: #2980b9;'>{MIN_SUPP}</span></p>
            <p>Minimum rule confidence: <span style='font-style: italic; color: #2980b9;'>{MIN_CONFIDENCE}</span></p>
        </div>
    </section>
    """

html_content += "</div>"
display(HTML(html_content))

In [ ]:
PREPROCESSING_ERROR = False

In [ ]:
try:
    # Load dataset
    df = pd.read_json(DATASET)
    df.index.name = "index"
    df.columns = [str(col) for col in df.columns]
    classes = df[CLASS_ATTRIBUTE].value_counts()
    samples_per_class = min(classes.min(), 5)
    sample = df.groupby(CLASS_ATTRIBUTE, group_keys=False).apply(lambda x: x.sample(samples_per_class))
    selected_indices = sample.index
    sample_will_change = DECISION_CLASS_BALANCING or not MISSING_VALUES_IMPUTATION
    display(md(f"## Dataset sample"))
    show(sample.reset_index(drop=True), show_index=False)
    
    # Preprocessing
    display(md("## Preprocessed dataset"))
    
    if UNIMPORTANT_ATTRIBUTE_ELIMINATION:
        low_correlation_attribute = {}
        low_quality_details = {}
        if USE_LOW_CORRELATION_ATTRIBUTE_REMOVAL: 
            df, low_correlation_attributes = remove_low_correlation_attributes(df, CLASS_ATTRIBUTE)
        ISMT_THRESHOLD = {'I': I, 'S': T, 'M': M, 'T': T}
        df, low_quality_details = remove_low_quality_attributes(df, CLASS_ATTRIBUTE, ISMT_THRESHOLD)
        display(md("#### Removed attributes"))
        reasons_combined = {**low_correlation_attributes, **low_quality_details}
        for reason, details in reasons_combined.items():
            if details is not None and len(details) > 0:
                if isinstance(details, pd.Index):
                    details = details.tolist()  
                details_str = ', '.join(details)
            else:
                details_str = 'None'
            display(md(f"**{reason}:** {details_str}"))
    
    if MISSING_VALUES_IMPUTATION:
        df = impute_missing_values(df, CLASS_ATTRIBUTE, STRATEGY_NUMERICAL, STRATEGY_NOMINAL)
    else:
        df = df.dropna()
    
    if DECISION_CLASS_BALANCING: 
        encoded_df, label_encoder, onehot_encoder, categorical_cols = encode_dataset(df, CLASS_ATTRIBUTE)
        encoded_df, DECISION_CLASS_BALANCING = balance_decision_classes(encoded_df, CLASS_ATTRIBUTE)
        df = decode_dataset(encoded_df, df, onehot_encoder, CLASS_ATTRIBUTE, categorical_cols, label_encoder)

    if sample_will_change:
        sample = df.groupby(CLASS_ATTRIBUTE, group_keys=False).apply(lambda x: x.sample(samples_per_class))
        selected_indices = sample.index
        display(md("NOTE: Due to decision class balancing and/or removal of missing values, preprocessed and encoded samples will contain different examples than the initial sample."))
    
    display(md("#### Dataset after preprocessing"))
    preprocessed_sample = df.loc[selected_indices].reset_index(drop=True)
    show(preprocessed_sample, show_index=False)
    
    encoded_df, label_encoder, onehot_encoder, categorical_cols = encode_dataset(df, CLASS_ATTRIBUTE)
    
    display(md("#### Dataset after one-hot encoding and label encoding"))
    encoded_sample = encoded_df.loc[selected_indices].reset_index(drop=True)
    show(encoded_sample, show_index=False)
except Exception as e:
    logger.exception(e)
    PREPROCESSING_ERROR = True
    display(md("An error occurred during preprocessing. No models will be trained."))

In [ ]:
if TREES and not PREPROCESSING_ERROR:
    display(md("## Trees"))
    try:
        build_and_show_classification_decision_tree(df, CLASS_ATTRIBUTE, max_depth=MAX_DEPTH)
    except Exception as e:
        logger.exception(e)
        display(md("An error occurred during tree generation."))

In [ ]:
if RULES and not PREPROCESSING_ERROR:
    display(md(f"## Rules"))
    try:
        display(md("### Rules generated with CN2 algorithm"))
        
        params = {
        "min_covered_examples": MIN_COVERED_EXAMPLES,
        "max_rule_length":  MAX_RULE_LENGTH,
        "beam_width": BEAM_WIDTH
        }
        cn2_model = CN2Model(params)
        X_train, y_train = cn2_model.prepare_data(df, test=None, class_name=CLASS_ATTRIBUTE)
        cn2_model.fit(X_train, y_train)
        rules = cn2_model.get_rules()
        for rule in rules:
            display(HTML(f"<div style='padding-left: 9em;'>{rule}</div>"))
        stats = cn2_model.get_stats()
        stats_df = pd.DataFrame(list(stats.items()), columns=['Statistic', 'Value'])
        display(md("#### Ruleset statistics:"))
        show(stats_df)
        predictions = cn2_model.predict(X_train)
        calculate_indicators(y_train, predictions)
    except Exception as e:
        logger.exception(e)
        display(md("An error occurred during rule generation."))

In [ ]:
if ITEMSETS_AND_ASSOCIATION and not PREPROCESSING_ERROR:
    try:
        frequent_itemsets = find_frequent_itemsets(df, CLASS_ATTRIBUTE, EXCLUDE_TARGET_COLUMN, MIN_SUPP, MAX_K_VALUE, MAX_INTERVALS_VALUE)
    
        display(md("## Frequent itemset mining"))
        if frequent_itemsets.empty:
            display(md("No frequent itemsets found"))
        else:
            display(md("**Frequent itemset mining was conducted on the dataset following one-hot encoding.**"))
            display(md(f"Showing top {MAX_ROWS} or fewer frequent itemsets with the highest support."))
            frequent_itemsets_sample = frequent_itemsets.sort_values(by='support', ascending=False).head(MAX_ROWS)
            show(frequent_itemsets_sample, showIndex=False, maxRows=MAX_ROWS, maxBytes=0)

        display(md("## Associations"))
        if frequent_itemsets.empty:
            display(md("No frequent itemsets found"))
            display(md("Cannot display associations without frequent itemsets"))
        else:
            display(md(f"Showing top {MAX_ROWS} or fewer associations with the highest lift, support, and confidence."))
            rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=MIN_CONFIDENCE)
            filtered_rules = rules[(rules['support'] >= MIN_SUPP) & (rules['confidence'] >= MIN_CONFIDENCE)]
            filtered_rules_sample = filtered_rules.sort_values(['lift', 'support', 'confidence'], ascending=False).head(MAX_ROWS)
            show(filtered_rules_sample[['antecedents', 'consequents', 'lift', 'support', 'confidence']], showIndex=False, maxRows=MAX_ROWS, maxBytes=0)
    except Exception as e:
        logger.exception(e)
        display(md("An error occurred during itemset mining and association rule generation."))

In [ ]:
if not PREPROCESSING_ERROR:
    try:
        display(md(f"## Attribute importance"))
        feature_importance_xgboost(df, CLASS_ATTRIBUTE, xgb.XGBClassifier)
    except Exception as e:
        logger.exception(e)
        display(md("An error occurred during attribute importance calculation."))